# W4: 채널별 매출 비교

스마트스토어/쿠팡 매출 데이터를 비교해 누적합을 막대그래프로 시각화합니다.


In [ ]:
import io
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

try:
    import google.generativeai as genai
except Exception:
    genai = None

GEMINI_MODEL = "gemini-2.0-flash"

for font_path in fm.findSystemFonts():
    try:
        fm.fontManager.addfont(font_path)
    except Exception:
        continue


def safe_upload():
    try:
        from google.colab import files
    except Exception:
        print("Colab 환경이 아니라 files.upload를 사용할 수 없습니다.")
        return {}

    try:
        return files.upload()
    except Exception as e:
        print(f"files.upload() 취소/실패: {e}")
        print("files.upload() 취소되어 기본 데이터 사용")
        return {}


def use_gemini(prompt: str, fallback: str) -> str:
    if not genai:
        return fallback
    key = os.getenv("GOOGLE_API_KEY", "").strip()
    if not key:
        return fallback
    try:
        genai.configure(api_key=key)
        model = genai.GenerativeModel(GEMINI_MODEL)
        out = model.generate_content(prompt)
        return getattr(out, "text", "").strip() or fallback
    except Exception as e:
        print(f"Gemini 호출 실패: {e}")
        return fallback

uploaded = safe_upload()
if uploaded:
    fn = list(uploaded.keys())[0]
    sales = pd.read_csv(io.StringIO(uploaded[fn].decode("utf-8")))
else:
    sales = pd.DataFrame(
        {
            "일자": pd.date_range("2026-01-01", periods=10, freq="D").astype(str),
            "채널": ["스마트스토어", "쿠팡", "스마트스토어", "쿠팡", "스마트스토어", "쿠팡", "스마트스토어", "쿠팡", "스마트스토어", "쿠팡"],
            "매출": [120000, 90000, 110000, 95000, 125000, 100000, 130000, 97000, 118000, 103000],
            "주문수": [120, 95, 130, 101, 110, 88, 128, 92, 112, 106]
        }
    )

by_channel = sales.groupby("채널", as_index=False)["매출"].sum()
fig, ax = plt.subplots(figsize=(6,3))
ax.bar(by_channel["채널"], by_channel["매출"], color=["#4c72b0", "#dd8452"])
ax.set_title("스마트스토어 vs 쿠팡 매출 비교")
plt.tight_layout()
plt.show()
print(by_channel)
